In [680]:
from typing import List, Tuple, Dict, Set, Any
from collections import defaultdict, deque
import heapq
import re
from copy import deepcopy

def read_input(filename: str) -> str:
    """Read input file and return contents as string."""
    with open(filename, 'r') as f:
        return f.read().strip()

def read_input_lines(filename: str) -> List[str]:
    """Read input file and return list of lines with whitespace stripped."""
    with open(filename, 'r') as f:
        return [line.strip() for line in f.readlines()]

def read_input_ints(filename: str) -> List[int]:
    """Read input file and return list of integers."""
    return [int(x) for x in read_input_lines(filename)]

def get_ints(text: str) -> List[int]:
    """Extract all integers from a string."""
    return [int(x) for x in re.findall(r'-?\d+', text)]

def manhattan_distance(p1: Tuple[int, int], p2: Tuple[int, int]) -> int:
    """Calculate Manhattan distance between two points."""
    return abs(p1[0] - p2[0]) + abs(p1[1] - p2[1])

def grid_to_dict(grid: List[List[Any]]) -> Dict[Tuple[int, int], Any]:
    """Convert 2D grid to dictionary with (row, col) coordinates as keys."""
    return {(i, j): val 
            for i, row in enumerate(grid)
            for j, val in enumerate(row)}

def get_adjacent(point: Tuple[int, int], include_diagonals: bool = False) -> List[Tuple[int, int]]:
    """Get adjacent coordinates (optionally including diagonals)."""
    x, y = point
    adjacent = [(x+1, y), (x-1, y), (x, y+1), (x, y-1)]
    if include_diagonals:
        adjacent.extend([(x+1, y+1), (x+1, y-1), (x-1, y+1), (x-1, y-1)])
    return adjacent

def bfs(start: Any, get_neighbors_fn, is_target_fn) -> Tuple[bool, Dict[Any, Any]]:
    """
    Breadth-first search returning if target found and parent dictionary for path reconstruction.
    
    Args:
        start: Starting node
        get_neighbors_fn: Function that returns neighbors for a given node
        is_target_fn: Function that returns True if node is target
    
    Returns:
        Tuple of (target_found, parent_dict)
    """
    queue = deque([start])
    visited = {start}
    parent = {}
    
    while queue:
        current = queue.popleft()
        
        if is_target_fn(current):
            return True, parent
            
        for neighbor in get_neighbors_fn(current):
            if neighbor not in visited:
                visited.add(neighbor)
                parent[neighbor] = current
                queue.append(neighbor)
                
    return False, parent

def dijkstra(start: Any, get_neighbors_with_costs_fn) -> Dict[Any, int]:
    """
    Dijkstra's algorithm for finding shortest paths.
    
    Args:
        start: Starting node
        get_neighbors_with_costs_fn: Function that returns list of (neighbor, cost) tuples
    
    Returns:
        Dictionary of distances to each node
    """
    distances = defaultdict(lambda: float('inf'))
    distances[start] = 0
    pq = [(0, start)]
    visited = set()
    
    while pq:
        current_dist, current = heapq.heappop(pq)
        
        if current in visited:
            continue
            
        visited.add(current)
        
        for neighbor, cost in get_neighbors_with_costs_fn(current):
            distance = current_dist + cost
            
            if distance < distances[neighbor]:
                distances[neighbor] = distance
                heapq.heappush(pq, (distance, neighbor))
                
    return distances

def reconstruct_path(parent: Dict[Any, Any], target: Any) -> List[Any]:
    """Reconstruct path from parent dictionary returned by search algorithms."""
    path = []
    current = target
    while current in parent:
        path.append(current)
        current = parent[current]
    path.append(current)
    return list(reversed(path))

def rotate_grid_clockwise(grid: List[List[Any]]) -> List[List[Any]]:
    """Rotate a 2D grid 90 degrees clockwise."""
    return list(zip(*grid[::-1]))

def find_cycle_length(sequence: List[Any]) -> Tuple[int, int]:
    """
    Find the prefix length and cycle length in a sequence using Floyd's algorithm.
    Returns tuple of (prefix_length, cycle_length).
    """
    # Find a repetition
    tortoise = 1
    hare = 2
    while sequence[tortoise] != sequence[hare]:
        tortoise += 1
        hare += 2
    
    # Find start of cycle
    mu = 0
    tortoise = 0
    while sequence[tortoise] != sequence[hare]:
        tortoise += 1
        hare += 1
        mu += 1
    
    # Find cycle length
    lam = 1
    hare = tortoise + 1
    while sequence[tortoise] != sequence[hare]:
        hare += 1
        lam += 1
    
    return mu, lam

In [681]:
def find_overlap(pattern,text):
    matches = [m.group(0) for m in re.finditer(pattern, text)]
    return matches


In [682]:
def digits(string):
    return re.findall(r'\d+',string)

The disk map uses a dense format to represent the layout of files and free space on the disk. The digits alternate between indicating the length of a file and the length of free space.

In [683]:
from functools import lru_cache
import re
from collections import defaultdict
from itertools import product

In [684]:

input = '''Register A: 729
Register B: 0
Register C: 0

Program: 0,1,5,4,3,0'''.strip().splitlines()
input = read_input_lines('input17.txt')

input = [[int(i) for i in digits(c)] for c in input]
n = len(input[:-1])
print(n)
print(input)

4
[[33940147], [0], [0], [], [2, 4, 1, 5, 7, 5, 1, 6, 4, 2, 5, 5, 0, 3, 3, 0]]


In [685]:
registers = {0: 0, 1: 0, 2:0}
pntr = [0]
std_out = []

In [686]:
def init():
    r = 0
    for k in input[:-2]:
        registers[r] = k[0]
        r+=1
    print('Initialising registers: ' + str(registers))

In [687]:
def combo(cmb):
    match(cmb):
        case 4: return registers[0]
        case 5: return registers[1]
        case 6: return registers[2]
        case _: return cmb

In [688]:
def adv(cmb):
    registers[0] = registers[0] // (2**combo(cmb))
    pntr[0] += 2
def bxl(literal): # tested
    registers[1] = registers[1] ^ literal
    pntr[0] += 2
def bst(cmb): # tested
    registers[1] = combo(cmb) % 8
    pntr[0] += 2
def jnz(literal):
    if registers[0] != 0:
        pntr[0] = literal
    else:
        pntr[0] += 2
def bxc(literal): # tested
    registers[1] = registers[1] ^ registers[2]
    pntr[0] += 2
def out(cmb):
    std_out.append(combo(cmb) % 8)
    pntr[0] += 2
def bdv(cmb):
    registers[1] = registers[0] // (2**combo(cmb))
    pntr[0] += 2
def cdv(cmb):
    registers[2] = registers[0] // (2**combo(cmb))
    pntr[0] += 2

instructions = {0: adv,1:bxl,2:bst,3:jnz,4:bxc,5:out,6:bdv,7:cdv} 




In [689]:
def print_binary_numbers(numbers):
    """
    Print a list of integers along with their binary representations.
    
    Args:
        numbers (list): A list of integers to convert to binary
        
    Prints:
        For each number, displays:
        - The original decimal number
        - Its binary representation
        - The length of the binary number
        - Statistics about ones and zeros
    """
    if not numbers:
        return
        
    # Find the longest number for padding
    max_length = max(len(str(num)) for num in numbers)
    st = ''
    for num in numbers:
        # Convert to binary and remove the '0b' prefix
        binary = bin(num)[2:]
        
        # Count ones and zeros
        ones = binary.count('1')
        zeros = binary.count('0')
        ones_percentage = (ones / len(binary)) * 100
        st += str(binary)
        st += ', '
    print(st)

In [690]:
def run():
    print('start run')
    while pntr[0] < len(input[-1]):
        inst = input[-1][pntr[0]]
        val = input[-1][pntr[0]+1]
        instructions[inst](val)
        print_binary_numbers(std_out)
    #print("Results: " + str(std_out))

In [691]:
print(''.join(str(s) for s in std_out))

In [692]:
def left_match_count(list1, list2):
    for i, (a, b) in enumerate(zip(list1, list2)):
        if a != b:
            return i
    return min(len(list1), len(list2))

In [693]:

#input[-1] = [0,3,5,4,3,0]
k = 0
lmatch = 0
n = len(input[-1])
print(n)
while k < 10:
    registers = {0:k,1:0,2:0}
    pntr = [0]
    std_out = []
    run()
    count = left_match_count(std_out,input[-1])
    if lmatch < count:
        print('new best match: ' + str(count) + ' at: ' + str(k))
        print(std_out)
        print(len(std_out))
        print(registers)
        lmatch=count
    k+=1


16
start run
11, 
11, 
11, 
start run
10, 
10, 
10, 
new best match: 1 at: 1
[2]
1
{0: 0, 1: 2, 2: 0}
start run
1, 
1, 
1, 
start run
0, 
0, 
0, 
start run
101, 
101, 
101, 
start run
11, 
11, 
11, 
start run
101, 
101, 
101, 
start run
101, 
101, 
101, 
start run
11, 
11, 
11, 
11, 
11, 
11, 
11, 
11, 
11, 10, 
11, 10, 
11, 10, 
start run
10, 
10, 
10, 
10, 
10, 
10, 
10, 
10, 
10, 10, 
10, 10, 
10, 10, 


In [694]:
k = 0
while k < len(input[-1]):
    print('[INST] ' + str(instructions[input[-1][k]]))
    print('[VAL] ' + str(input[-1][k+1]))
    k+=2

[INST] <function bst at 0x10d47f9c0>
[VAL] 4
[INST] <function bxl at 0x10d47d3a0>
[VAL] 5
[INST] <function cdv at 0x10d13c680>
[VAL] 5
[INST] <function bxl at 0x10d47d3a0>
[VAL] 6
[INST] <function bxc at 0x10d47fec0>
[VAL] 2
[INST] <function out at 0x10d13fec0>
[VAL] 5
[INST] <function adv at 0x10d47e0c0>
[VAL] 3
[INST] <function jnz at 0x10d47e840>
[VAL] 0


In [705]:

def test(b):
    
    u = []
    def it(a):
        b = a % 8
        b = b ^ 5
        print(b)
        c = a >> b
        b = b ^ 6
        b = b ^ c 
        u.append(b % 8)
        return a >> 3
    a = 33940147
    while (a > 0):
        a = it(a)
    print(u)

test(1)

6
3
7
4
3
6
4
5
7
[2, 7, 6, 5, 6, 0, 2, 3, 1]


In [761]:
def program_step(a):
    b = a % 8
    b = b ^ 5
    c = a >> b
    b = b ^ 6
    b = b ^ c 
    return b % 8

In [762]:
res = []
out = [2, 4, 1, 5, 7, 5, 1, 6, 4, 2, 5, 5, 0, 3, 3, 0]
def reverse_search(a,i):
    if i == 0:
        res.append(a)
        return 

    a_before = [i for i in range(a << 3, (a<<3)+8)]
    b_target = out[i-1]

    for a in a_before:
        if program_step(a) == b_target:
            reverse_search(a,i-1)
reverse_search(0,len(out))
print(min(res))

107416870455451
